0. Intialisation And Import

In [3]:

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import os
os.chdir(r'C:\Users\benjo\Documents\Projects\ecg-risk-stratification')
print(os.getcwd())

import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


C:\Users\benjo\Documents\Projects\ecg-risk-stratification


In [4]:
binary_df = pd.read_pickle('data/ptbxl_binary_metadata.pkl')
binary_df.head(10)


,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,diagnostic_superclass,label
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr,[NORM],0
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr,[NORM],0
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr,[NORM],0
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr,[NORM],0
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr,[NORM],0
6,19005.0,18.0,1,NaN,58.0,2.0,0.0,CS-12 E,1984-11-28 13:32:13,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,4,records100/00000/00006_lr,records500/00000/00006_hr,[NORM],0
7,16193.0,54.0,0,NaN,83.0,2.0,0.0,CS-12 E,1984-11-28 13:32:22,"sinusrhythmus linkstyp t abnormal, wahrscheinl...",...,NaN,NaN,NaN,NaN,NaN,7,records100/00000/00007_lr,records500/00000/00007_hr,[NORM],0
9,18792.0,55.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-12-08 09:44:43,sinusrhythmus normales ekg,...,", I-AVR,",NaN,NaN,NaN,NaN,10,records100/00000/00009_lr,records500/00000/00009_hr,[NORM],0
10,9456.0,22.0,1,NaN,56.0,2.0,0.0,CS-12 E,1984-12-12 14:12:46,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,NaN,9,records100/00000/00010_lr,records500/00000/00010_hr,[NORM],0


Caching of ECGs to improve loading speed later for training (discovered this was a signigicant time bottleneck when first training)

In [7]:
DATA_DIR = 'data/'
CACHE_DIR = 'data/preprocessed_500'

from src.preprocessing import preprocess_and_cache

preprocess_and_cache(
    df=binary_df,
    data_path=DATA_DIR,
    sampling_rate=500,
    use_filt=True,
    low=0.5,
    high=40,
)

1. Splitting of the Data into training validation and testing folds, this is provided by the dataset itself

In [5]:
train_df = binary_df[binary_df["strat_fold"].isin(range(1, 9))].copy()
val_df = binary_df[binary_df["strat_fold"] == 9].copy()
test_df = binary_df[binary_df["strat_fold"] == 10].copy()
#Testing for Leakage

train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])
test_patients = set(test_df["patient_id"])


print(len(train_patients & val_patients))
print(len(train_patients & test_patients))
print(len(val_patients & test_patients))#Note: in sets the & is the intersection and length shiows how much overlap

0
0
0


In [4]:
print("Train labels:")
print(train_df["label"].value_counts(normalize=True))
print("Val labels:")
print(val_df["label"].value_counts(normalize=True))
print("Test labels:")
print(test_df["label"].value_counts(normalize=True))

Train labels:
label
0    0.633739
1    0.366261
Name: proportion, dtype: float64
Val labels:
label
0    0.633842
1    0.366158
Name: proportion, dtype: float64
Test labels:
label
0    0.636427
1    0.363573
Name: proportion, dtype: float64


2. Pre-Processing

In [5]:
from src.preprocessing import load_and_preprocess
from src.data_extraction import load_ecg

ecg_id = binary_df.index[0]

signal, metadata, row = load_and_preprocess(
    ecg_id,
    binary_df,
    "data/",
    sampling_rate=500
)

print("ECG ID:", ecg_id)
print("Signal shape:", signal.shape)
print("Sampling frequency:", metadata["fs"])
print("Lead names:", metadata["sig_name"])
print("Label:", row["label"])

print("Mean per lead:")
print(signal.mean(axis=1))

print("Std per lead:")
print(signal.std(axis=1))

ECG ID: 1
Signal shape: (12, 5000)
Sampling frequency: 500
Lead names: ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
Label: 0
Mean per lead:
[ 1.3732910e-08 -3.0517577e-09  6.1035155e-09  9.9182129e-09
 -2.2888185e-09  8.3923339e-09  3.0517577e-09  5.3405760e-09
  1.5258789e-09 -2.2888185e-09  4.5776369e-09  1.6403199e-08]
Std per lead:
[1.         1.         0.9999997  0.9999999  0.99999976 0.9999997
 0.99999994 0.9999998  0.99999994 0.9999999  0.9999999  0.99999976]


3. Creating the Correct Dataset

In [8]:
from src.dataset import ECGdataset
'''
train_dataset = ECGdataset(
    train_df,
    "data/",
    samp_rate=500,
    use_filt=True,
)

test_dataset = ECGdataset(
    test_df,
    "data/",
    samp_rate=500,
    use_filt=True,
)

val_dataset = ECGdataset(
    val_df,
    "data/",
    samp_rate=500,
    use_filt=True,
)

WITH ADDITION OF CACHING, THIS IS NOW UNNECESSARY AND REPLACED BY BELOW CODE

'''

from src.dataset import CachedECGDataset
train_dataset = CachedECGDataset(train_df, cache_dir=CACHE_DIR)
val_dataset = CachedECGDataset(val_df, cache_dir=CACHE_DIR)
test_dataset = CachedECGDataset(test_df, cache_dir=CACHE_DIR)



4. Dataloader creation 

In [9]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
)


5. Model Training 

In [11]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.model import simpleECGNN
from src.training import train_an_epoch, eval

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
model = simpleECGNN(dropout=0.3).to(device)


In [39]:
os.makedirs("results/models", exist_ok=True)
os.makedirs("results/plots", exist_ok=True)
os.makedirs("results/metrics", exist_ok=True)

num_pos = train_df["label"].sum()
num_neg = len(train_df) - num_pos
pos_weight = torch.tensor(num_neg / num_pos, dtype=torch.float32).to(device)


criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3,weight_decay=1e-5)

In [ ]:
from src.training import train

history = train(
    model,
    optimizer,
    criterion,
    train_loader,
    val_loader,
    device,
    num_epochs=30,
    patience=3,
    tolerance=1e-4
)

history_df = pd.DataFrame(history)
history_df.to_csv("results/metrics/training_history.csv", index=False)

Batch 0/179 - Loss: 0.2089
Batch 20/179 - Loss: 0.3870
Batch 40/179 - Loss: 0.3572
Batch 60/179 - Loss: 0.2900
Batch 80/179 - Loss: 0.3630
Batch 100/179 - Loss: 0.2694
Batch 120/179 - Loss: 0.2875
Batch 140/179 - Loss: 0.2942
Batch 160/179 - Loss: 0.3287
Epoch 1/30 - Train Loss: 0.2681 - Val Loss: 0.2452 - Val AUROC: 0.9771 - Val AUPRC: 0.9614 - Val F1: 0.8976
New best model saved with AUROC: 0.9771
Batch 0/179 - Loss: 0.3047
Batch 20/179 - Loss: 0.1093
Batch 40/179 - Loss: 0.1807
Batch 60/179 - Loss: 0.1988
Batch 80/179 - Loss: 0.4928
Batch 100/179 - Loss: 0.2875
Batch 120/179 - Loss: 0.2587
Batch 140/179 - Loss: 0.2148
Batch 160/179 - Loss: 0.2243
Epoch 2/30 - Train Loss: 0.2709 - Val Loss: 0.2518 - Val AUROC: 0.9769 - Val AUPRC: 0.9605 - Val F1: 0.8889
Batch 0/179 - Loss: 0.2471
Batch 20/179 - Loss: 0.2293
Batch 40/179 - Loss: 0.2106
Batch 60/179 - Loss: 0.3305
Batch 80/179 - Loss: 0.3557
Batch 100/179 - Loss: 0.3031
Batch 120/179 - Loss: 0.2915
Batch 140/179 - Loss: 0.4601
Batch 16

Bottle Neck Finding in Terms of Time

In [34]:
import time

start = time.time()

x, y = next(iter(train_loader))

end = time.time()

print("Time to load one batch:", end - start, "seconds")
print("x shape:", x.shape)
print("y shape:", y.shape)

Time to load one batch: 0.730811595916748 seconds
x shape: torch.Size([16, 12, 5000])
y shape: torch.Size([16])


In [56]:
num_batches = 20

start = time.time()

for batch_idx, (x, y) in enumerate(train_loader):
    if batch_idx >= num_batches:
        break

end = time.time()

print(f"Loaded {num_batches} batches in {end - start:.2f} seconds")
print(f"Average per batch: {(end - start) / num_batches:.2f} seconds")

Loaded 20 batches in 30.13 seconds
Average per batch: 1.51 seconds


In [36]:
x, y = next(iter(train_loader))

x = x.to(device).float()
y = y.to(device).float()

# Warm-up
for _ in range(5):
    logits = model(x)
    loss = criterion(logits, y)

if device.type == "cuda":
    torch.cuda.synchronize()

start = time.time()

num_runs = 50

for _ in range(num_runs):
    logits = model(x)
    loss = criterion(logits, y)

if device.type == "cuda":
    torch.cuda.synchronize()

end = time.time()

print(f"{num_runs} forward passes took {end - start:.2f} seconds")
print(f"Average forward pass: {(end - start) / num_runs:.4f} seconds")

50 forward passes took 3.33 seconds
Average forward pass: 0.0666 seconds


In [ ]:
x, y = next(iter(train_loader))

x = x.to(device).float()
y = y.to(device).float()

logits = model(x)
loss = criterion(logits, y)
loss.backward()
optimizer.step()
optimizer.zero_grad()

if device.type == "cuda":
    torch.cuda.synchronize()

num_runs = 20
start = time.time()

for _ in range(num_runs):
    optimizer.zero_grad()

    logits = model(x)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

if device.type == "cuda":
    torch.cuda.synchronize()

end = time.time()

print(f"{num_runs} full training steps took {end - start:.2f} seconds")
print(f"Average training step: {(end - start) / num_runs:.4f} seconds")

20 full training steps took 2.92 seconds
Average training step: 0.1458 seconds


In [ ]:
raw_dataset = ECGdataset(
    train_df,
    "data/",
    samp_rate=500,
    use_filt=True,
)

cached_dataset = CachedECGDataset(
    train_df,
    cache_dir="data/preprocessed_500",
)

idx = 0

start = time.time()
x, y = raw_dataset[idx]
print("Raw single sample:", time.time() - start)

start = time.time()
x, y = cached_dataset[idx]
print("Cached single sample:", time.time() - start)

Raw single sample: 0.0782921314239502
Cached single sample: 0.004855155944824219


In [ ]:
workers = [0]
for n in workers:
    raw_loader = DataLoader(
        raw_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=n,
    )

    cached_loader = DataLoader(
        cached_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=n,
    )

    start = time.time()
    for i, (x, y) in enumerate(raw_loader):
        if i >= 20:
            break
    print("Raw 20 batches:", time.time() - start)

    start = time.time()
    for i, (x, y) in enumerate(cached_loader):
        if i >= 20:
            break
    print("Cached 20 batches:", time.time() - start)

Raw 20 batches: 107.83544278144836
Cached 20 batches: 4.143548965454102
Raw 20 batches: 92.58464550971985
Cached 20 batches: 27.309333086013794
Raw 20 batches: 54.18491005897522
Cached 20 batches: 38.40159559249878
Raw 20 batches: 133.42112040519714
Cached 20 batches: 111.51269769668579


In [61]:
batch_sizes = [16, 32, 64, 128]
for n in batch_sizes:
    cached_loader = DataLoader(
        cached_dataset,
        batch_size=64,
        shuffle=True,
        num_workers=0,
    )
    start = time.time()
    for i, (x, y) in enumerate(cached_loader):
        if i >= 20:
            break
    print("Cached 20 batches:", time.time() - start)

Cached 20 batches: 12.894479751586914
Cached 20 batches: 12.065725564956665
Cached 20 batches: 9.795295238494873
Cached 20 batches: 10.051141262054443


6. Training Curves